## Introduction:

I chose the Arlington Real Estate Assessment Payment Summary dataset because it has the financial data of real estate taxes and payment summaries in Arlington County. It's interesting to analyze for trends in property tax totals.
* **Link to dataset page and API documentation:** 
  * [Arlington Open Data Portal](https://datahub-v2.arlingtonva.us/)
  * API Endpoint Used: https://datahub-v2.arlingtonva.us/api/Tax/RealEstateAssessmentPaymentSummary


In [4]:
import requests
import matplotlib.pyplot as plt
import json
import pandas as pd

In [ ]:
base_url = 'https://datahub-v2.arlingtonva.us/api/Tax/RealEstateAssessmentPaymentSummary?'

response = requests.get(base_url) 

data = response.json() 

print(json.dumps(data, indent=2))


## Explore the Data:

**Data Structure:** The data returned from the API is structured as a JSON list

**What does a single object represent?** Each item is an object that represents a unique real estate property's assessment and tax payment summary record in Arlington County



In [ ]:
base_url = 'https://datahub-v2.arlingtonva.us/api/Tax/RealEstateAssessmentPaymentSummary?'
args = '$top=2'
full_url = base_url + args

# Make the request and parse JSON
response = requests.get(full_url)
sample_data = response.json()

# Print the results formatted
print(json.dumps(sample_data, indent=2))

Based on the API response keys, here are the main attributes and what they mean:
1. **realEstatePropertyCode:** A unique identifier code assigned to each parcel of land or property in Arlington County
2. **taxYearDate:** The year associated with the real estate assessment and tax payment summary
3. **totalAmt** The total amount paid toward the real estate assessment/taxes for that property


### Question 1: What are the payment summaries for properties in ascending order by year?

In [ ]:
args = "$orderby=taxYearDate asc&$top=20"
full_url = base_url + args

response = requests.get(full_url)
data = response.json()

print(json.dumps(data, indent=2))



**What I learned:** Ordering taxYeardate by asc allowed me to see the properties in ascending order by date


### Question 2: What do the records look like when sorted in descending order by the total amount?

In [ ]:
args = "$orderby=totalAmt desc&$top=20"
full_url = base_url + args

response = requests.get(full_url)
data = response.json()

# Print the result nicely formatted as a JSON list
print(json.dumps(data, indent=2))

**What I learned:** Using $orderby in descending order allows me to look at highest-value assessment summaries at the top of the financial spectrum

### Question 3: Are there any records where the total amount is negative, indicating an account credit or adjustment?

In [ ]:
# Filter for total amount less than zero ('lt 0') to find credits/adjustments
args = "$filter=totalAmt lt 0&$top=20"
full_url = base_url + args

response = requests.get(full_url)
data = response.json()

# Print the result nicely formatted as a JSON list
print(json.dumps(data, indent=2))

**What I learned:** By filtering for negative totalAmt values, financial refunds or adjustments and be isolated

## Visualizations:


In [ ]:

# 1. Fetch a larger dataset sorted by ascending year
base_url = 'https://datahub-v2.arlingtonva.us/api/Tax/RealEstateAssessmentPaymentSummary?'
args = '$top=10000&orderby=taxYearDate asc'
full_url = base_url + args

response = requests.get(full_url)
data = response.json()

# 2. Convert the JSON list into a pandas DataFrame
df = pd.DataFrame(data)

# 3. Count how many records exist for each taxYearDate and sort by the year index
year_counts = df['taxYearDate'].value_counts().sort_index()

# 4. Create the Matplotlib line chart
plt.figure(figsize=(12, 6))
plt.plot(year_counts.index, year_counts.values, marker='o', color='purple', linestyle='-', linewidth=2)

# Add titles, labels, and grid for readability
plt.title('Number of Real Estate Assessment Records per Year (1974–2026)')
plt.xlabel('Tax Year (taxYearDate)')
plt.ylabel('Number of Records / Properties')
plt.grid(True)
plt.show()



**What this chart shows:** The distribution of real estate assessment payment summary records logged for each year from 1974 through 2026

**Why I chose it:** I chose a line chart because it makes it easy to spot trends across a wide timeline


In [ ]:
# 1. Fetch data from the API
base_url = 'https://datahub-v2.arlingtonva.us/api/Tax/RealEstateAssessmentPaymentSummary?'
args = '$top=50000'
response = requests.get(base_url + args)
df = pd.DataFrame(response.json())

# 2. Convert totalAmt to numeric and take absolute values (to handle credits/negative adjustments)
df['totalAmt'] = pd.to_numeric(df['totalAmt'], errors='coerce').abs()

# 3. Define your custom bins and labels
bins = [0, 1000, 2000, 3000, 4000, 5000, 6000, 7000, 8000, 9000, 10000]
labels = ['$0-$1,000', '$1,001-$2,000', '$2,001-$3,000', '$3,001-$4,000', '$4,001-$5,000', '$5,001-$6,000', '$6,001-$7,000', '$7,001-$8,000', '$8,001-$9,000', '$9,001-$10,000']

# Categorize the data into the specified ranges using pd.cut
df['amount_range'] = pd.cut(df['totalAmt'], bins=bins, labels=labels)

# Count how many records fall into each range
range_counts = df['amount_range'].value_counts().sort_index()

# 4. Create the Bar Chart
plt.figure(figsize=(10, 6))
plt.bar(range_counts.index, range_counts.values, color='cornflowerblue', edgecolor='black')

# Add titles, labels, and grid
plt.title('Number of Properties by Payment Amount Ranges')
plt.xlabel('Payment Amount Ranges')
plt.ylabel('Number of Properties / Records')
plt.xticks(rotation=25)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()


**What this chart shows:** This bar graph categorizes property records into specific financial tiers ranging from $0–$1000 up to $9000–$10000

**Why I chose it:** I chose a bar chart because it provides a clear comparison of how financial values are distributed across different tiers. This makes it easy to see which payment range is most common in Arlington County.
